In [1]:
import pandas as pd

# Dataset paths
metadata_path = "/kaggle/input/datasets/mdnaimislam165436/jaal-taka/JaalTaka Metadata.csv"

# Load metadata
df = pd.read_csv(metadata_path)

# Clean column names and relevant values
df.columns = df.columns.str.strip()
df["Type"] = df["Type"].astype(str).str.strip().str.lower()
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# Count UNIQUE notes based on Note serial
unique_notes = (
    df.groupby(["Type", "Amount"])["Note serial"]
      .nunique()
      .unstack(fill_value=0)
)

print("Unique note count:")
print(unique_notes)

# More readable output
print("\n" + "=" * 50)
print("UNIQUE NOTE COUNT")
print("=" * 50)

for note_type in ["real", "fake"]:
    for amount in [500, 1000]:
        count = unique_notes.get(amount, pd.Series()).get(note_type, 0)
        print(f"{note_type.upper():5} | {amount} Taka | {count} unique notes")

Unique note count:
Amount  500   1000
Type              
fake     326   262
real     398   404

UNIQUE NOTE COUNT
REAL  | 500 Taka | 398 unique notes
REAL  | 1000 Taka | 404 unique notes
FAKE  | 500 Taka | 326 unique notes
FAKE  | 1000 Taka | 262 unique notes


In [2]:
import os
import shutil
import pandas as pd

# ============================================================
# Paths
# ============================================================

BASE_PATH = "/kaggle/input/datasets/mdnaimislam165436/jaal-taka/JaalTaka"

REAL_PATH = os.path.join(BASE_PATH, "real_notes")
FAKE_PATH = os.path.join(BASE_PATH, "fake_notes")

METADATA_PATH = "/kaggle/input/datasets/mdnaimislam165436/jaal-taka/JaalTaka Metadata.csv"

# Kaggle output directory
OUTPUT_PATH = "/kaggle/working/jaal_taka_actual"


# ============================================================
# Load metadata
# ============================================================

df = pd.read_csv(METADATA_PATH)

# Clean column names
df.columns = df.columns.str.strip()

# Clean values
df["Type"] = df["Type"].astype(str).str.strip().str.lower()
df["Note serial"] = df["Note serial"].astype(str).str.strip()
df["Taka serial"] = df["Taka serial"].astype(str).str.strip()


# ============================================================
# Keep only ONE row for each unique Taka serial
# ============================================================
# Taka serial = unique physical banknote

unique_df = df.drop_duplicates(
    subset=["Type", "Taka serial"]
).copy()

print("Total metadata rows:", len(df))
print("Unique physical notes:", len(unique_df))


# ============================================================
# Create output directories
# ============================================================

os.makedirs(os.path.join(OUTPUT_PATH, "real"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "fake"), exist_ok=True)


# ============================================================
# Copy unique note folders
# ============================================================

copied = 0
missing = 0

for _, row in unique_df.iterrows():

    note_type = row["Type"]
    note_serial = row["Note serial"]

    # Determine source directory
    if note_type == "real":
        source_root = REAL_PATH
    elif note_type == "fake":
        source_root = FAKE_PATH
    else:
        print(f"Skipping unknown type: {note_type}")
        continue

    source_folder = os.path.join(source_root, note_serial)

    # Destination
    destination_folder = os.path.join(
        OUTPUT_PATH,
        note_type,
        note_serial
    )

    # Check source folder
    if not os.path.isdir(source_folder):
        print(f"WARNING: Folder not found: {source_folder}")
        missing += 1
        continue

    # Copy entire folder including all images
    shutil.copytree(
        source_folder,
        destination_folder,
        dirs_exist_ok=True
    )

    copied += 1


# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 60)
print("DATASET CREATION COMPLETE")
print("=" * 60)

print(f"Unique notes found : {len(unique_df)}")
print(f"Folders copied     : {copied}")
print(f"Missing folders    : {missing}")

print("\nOutput:")
print(OUTPUT_PATH)

Total metadata rows: 1390
Unique physical notes: 426

DATASET CREATION COMPLETE
Unique notes found : 426
Folders copied     : 426
Missing folders    : 0

Output:
/kaggle/working/jaal_taka_actual


In [3]:
# ============================================================
# Verify number of images per note
# ============================================================

valid_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

print("\n" + "=" * 60)
print("VERIFYING OUTPUT DATASET")
print("=" * 60)

for note_type in ["real", "fake"]:

    type_path = os.path.join(OUTPUT_PATH, note_type)

    note_folders = [
        f for f in os.listdir(type_path)
        if os.path.isdir(os.path.join(type_path, f))
    ]

    print(f"\n{note_type.upper()}")
    print(f"Number of unique notes: {len(note_folders)}")

    image_counts = {}

    for note_folder in note_folders:

        folder_path = os.path.join(type_path, note_folder)

        images = [
            f for f in os.listdir(folder_path)
            if os.path.splitext(f)[1].lower() in valid_extensions
        ]

        image_counts[note_folder] = len(images)

    # Distribution of image counts
    from collections import Counter

    distribution = Counter(image_counts.values())

    print("Images per note distribution:")
    for count, number_of_notes in sorted(distribution.items()):
        print(f"  {count} images : {number_of_notes} notes")

    # Show notes that don't have exactly 6 images
    incorrect = {
        note: count
        for note, count in image_counts.items()
        if count != 6
    }

    if incorrect:
        print("\nWARNING - Notes without exactly 6 images:")
        for note, count in incorrect.items():
            print(f"  {note}: {count} images")
    else:
        print("✓ All notes contain exactly 6 images.")


VERIFYING OUTPUT DATASET

REAL
Number of unique notes: 399
Images per note distribution:
  6 images : 399 notes
✓ All notes contain exactly 6 images.

FAKE
Number of unique notes: 27
Images per note distribution:
  6 images : 27 notes
✓ All notes contain exactly 6 images.
